# Experiment 3: Error Handling and Logging (FastAPI)

**Objective:**
- Implement exception handlers in the FastAPI application
- Add structured logging for requests, responses, and errors
- Create a robust error handling middleware

**Prerequisites:** Run Experiments 1 and 2 first.

## Step 1: Install Required Libraries

In [ ]:
# VIVA NOTE: Install logging library for production-grade error tracking
# - python-json-logger: formats logs as JSON (machine-readable, easy to parse with ELK stack)
# - In production, structured JSON logs enable: log aggregation, search, alerting, dashboards
# - JSON logs > plain text logs for microservices and cloud deployments
import sys
!{sys.executable} -m pip install fastapi uvicorn pydantic scikit-learn pandas numpy nest-asyncio python-json-logger

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


## Step 2: Import Libraries

In [ ]:
# VIVA NOTE: Import production libraries for logging and error handling
# - logging: Python's standard logging framework
# - json: serialize logs to JSON format
# - uuid: generate unique request IDs (for tracing requests across services)
# - traceback: capture full error stack traces
# - datetime: timestamp each log entry (UTC for global consistency)
# - RequestValidationError: Pydantic validation errors
# - BaseHTTPMiddleware: intercept all requests/responses for logging
import pickle
import numpy as np
import pandas as pd
import uvicorn
import nest_asyncio
import threading
import time
import requests
import logging
import json
import uuid
import traceback
from datetime import datetime

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from pydantic import BaseModel, Field
from typing import Literal, Optional
from starlette.middleware.base import BaseHTTPMiddleware

nest_asyncio.apply()
print("All libraries imported successfully!")

All libraries imported successfully!


/Users/harshsmac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Step 3: Setup Structured Logging

In [ ]:
# VIVA NOTE: Setup structured JSON logging (production best practice)
# - JSONFormatter: custom formatter converts logs to JSON (timestamp, level, message, extras)
# - Extra fields: request_id (trace specific requests), method, url, status_code, duration_ms, error_detail
# - Three separate loggers: requests (all API calls), errors (only failures), app (business logic)
# - File handlers: persist logs to disk (rotate in production with RotatingFileHandler)
# - Console handlers: print to stdout (useful for Docker/Kubernetes where logs go to container runtime)
# - Why JSON? Easy parsing by log aggregation tools (ELK, Splunk, Datadog)
import os
os.makedirs('logs', exist_ok=True)

# Custom JSON Formatter
class JSONFormatter(logging.Formatter):
    def format(self, record):
        log_entry = {
            "timestamp": datetime.utcnow().isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        # Add extra fields if present
        if hasattr(record, 'request_id'):
            log_entry['request_id'] = record.request_id
        if hasattr(record, 'method'):
            log_entry['method'] = record.method
        if hasattr(record, 'url'):
            log_entry['url'] = record.url
        if hasattr(record, 'status_code'):
            log_entry['status_code'] = record.status_code
        if hasattr(record, 'duration_ms'):
            log_entry['duration_ms'] = record.duration_ms
        if hasattr(record, 'error_detail'):
            log_entry['error_detail'] = record.error_detail
        if record.exc_info:
            log_entry['exception'] = self.formatException(record.exc_info)
        return json.dumps(log_entry)

# Setup loggers
def setup_logger(name, log_file, level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.handlers = []  # Clear existing handlers
    
    # File handler with JSON format
    file_handler = logging.FileHandler(log_file)
    file_handler.setFormatter(JSONFormatter())
    logger.addHandler(file_handler)
    
    # Console handler
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(JSONFormatter())
    logger.addHandler(console_handler)
    
    return logger

# Create loggers
request_logger = setup_logger('request_logger', 'logs/requests.log')
error_logger = setup_logger('error_logger', 'logs/errors.log', logging.ERROR)
app_logger = setup_logger('app_logger', 'logs/app.log')

print("Logging setup completed!")
print("Log files: logs/requests.log, logs/errors.log, logs/app.log")

Logging setup completed!
Log files: logs/requests.log, logs/errors.log, logs/app.log


## Step 4: Load Model Artifacts

In [ ]:
# VIVA NOTE: Load model artifacts and log the operation
# - Same loading code as Experiment 2
# - app_logger.info(): logs successful model loading (helps debug startup issues)
# - In production: add try-except here to catch missing files early
with open('model_artifacts/churn_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('model_artifacts/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('model_artifacts/label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)
with open('model_artifacts/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

app_logger.info("Model artifacts loaded successfully")
print("Model artifacts loaded!")

{"timestamp": "2026-02-23T09:17:48.604173", "level": "INFO", "logger": "app_logger", "message": "Model artifacts loaded successfully"}


Model artifacts loaded!


## Step 5: Define Pydantic Schemas with Error Models

In [ ]:
# VIVA NOTE: Define Pydantic schemas including error responses
# - ChurnPredictionResponse now includes request_id (for request tracing)
# - ErrorResponse: standard error format with error type, detail, request_id, timestamp
# - Consistent error format helps API consumers handle errors gracefully
# - request_id links errors back to specific requests in logs
# Request schema
class ChurnPredictionRequest(BaseModel):
    Gender: Literal['Male', 'Female'] = Field(..., description="Customer gender")
    SeniorCitizen: int = Field(..., ge=0, le=1, description="Senior citizen (0 or 1)")
    Tenure: int = Field(..., ge=0, description="Months of tenure")
    MonthlyCharges: float = Field(..., ge=0, description="Monthly charges")
    Contract: Literal['Month-to-month', 'One year', 'Two year'] = Field(..., description="Contract type")
    PaymentMethod: Literal['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'] = Field(..., description="Payment method")
    TotalCharges: float = Field(..., ge=0, description="Total charges")

class ChurnPredictionResponse(BaseModel):
    prediction: int
    prediction_label: str
    churn_probability: float
    no_churn_probability: float
    request_id: str

# Error response schema
class ErrorResponse(BaseModel):
    error: str
    detail: str
    request_id: Optional[str] = None
    timestamp: str

print("Schemas defined!")

Schemas defined!


## Step 6: Create FastAPI App with Logging Middleware & Exception Handlers

In [ ]:
# VIVA NOTE: Logging Middleware & Exception Handlers (CORE PRODUCTION FEATURES)
#
# === LOGGING MIDDLEWARE ===
# - LoggingMiddleware: intercepts EVERY request before it reaches endpoint
# - Generates unique request_id for each request (tracing across microservices)
# - Logs: incoming request (method, URL), outgoing response (status, duration)
# - duration_ms: performance monitoring (identify slow endpoints)
# - X-Request-ID header: client can use this ID to report issues
# - try-except: catches unhandled errors, logs them with full traceback
#
# === EXCEPTION HANDLERS ===
# 1. RequestValidationError (422): Pydantic validation failures
#    - Field-level errors (wrong type, invalid value, missing field)
#    - Returns structured error list: [{field, message, type}]
# 
# 2. HTTPException: Explicit errors raised by endpoints (e.g., raise HTTPException(404))
#    - Logs error with request_id
#    - Returns JSON with error, detail, request_id, timestamp
#
# 3. Generic Exception (500): Unexpected errors (e.g., database crash, out of memory)
#    - Logs full traceback to error_logger
#    - Returns generic message (don't expose internal errors to clients)
#
# WHY THIS MATTERS FOR VIVA:
# - Production APIs MUST handle errors gracefully (no 500 crashes)
# - Structured logs enable debugging in production
# - Request IDs enable distributed tracing
# Create FastAPI app
app = FastAPI(
    title="Bank Churn Prediction API (with Logging)",
    description="API with structured logging and error handling",
    version="2.0.0"
)

# ====================== LOGGING MIDDLEWARE ======================
class LoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        request_id = str(uuid.uuid4())[:8]
        request.state.request_id = request_id
        start_time = time.time()
        
        # Log incoming request
        request_logger.info(
            f"Incoming request",
            extra={
                'request_id': request_id,
                'method': request.method,
                'url': str(request.url)
            }
        )
        
        try:
            response = await call_next(request)
            duration_ms = round((time.time() - start_time) * 1000, 2)
            
            # Log response
            request_logger.info(
                f"Request completed",
                extra={
                    'request_id': request_id,
                    'method': request.method,
                    'url': str(request.url),
                    'status_code': response.status_code,
                    'duration_ms': duration_ms
                }
            )
            response.headers['X-Request-ID'] = request_id
            return response
        except Exception as e:
            duration_ms = round((time.time() - start_time) * 1000, 2)
            error_logger.error(
                f"Unhandled error: {str(e)}",
                extra={
                    'request_id': request_id,
                    'method': request.method,
                    'url': str(request.url),
                    'duration_ms': duration_ms,
                    'error_detail': traceback.format_exc()
                }
            )
            raise

app.add_middleware(LoggingMiddleware)

# ====================== EXCEPTION HANDLERS ======================

# Handle validation errors
@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    request_id = getattr(request.state, 'request_id', 'unknown')
    error_details = []
    for error in exc.errors():
        error_details.append({
            "field": " -> ".join(str(loc) for loc in error['loc']),
            "message": error['msg'],
            "type": error['type']
        })
    
    error_logger.error(
        f"Validation error",
        extra={'request_id': request_id, 'error_detail': str(error_details)}
    )
    
    return JSONResponse(
        status_code=422,
        content={
            "error": "Validation Error",
            "detail": error_details,
            "request_id": request_id,
            "timestamp": datetime.utcnow().isoformat()
        }
    )

# Handle HTTP exceptions
@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    request_id = getattr(request.state, 'request_id', 'unknown')
    error_logger.error(
        f"HTTP error: {exc.status_code}",
        extra={'request_id': request_id, 'error_detail': exc.detail}
    )
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "error": f"HTTP {exc.status_code}",
            "detail": exc.detail,
            "request_id": request_id,
            "timestamp": datetime.utcnow().isoformat()
        }
    )

# Handle generic exceptions
@app.exception_handler(Exception)
async def generic_exception_handler(request: Request, exc: Exception):
    request_id = getattr(request.state, 'request_id', 'unknown')
    error_logger.error(
        f"Internal server error: {str(exc)}",
        extra={
            'request_id': request_id,
            'error_detail': traceback.format_exc()
        }
    )
    return JSONResponse(
        status_code=500,
        content={
            "error": "Internal Server Error",
            "detail": "An unexpected error occurred. Check logs for details.",
            "request_id": request_id,
            "timestamp": datetime.utcnow().isoformat()
        }
    )

print("Middleware and exception handlers configured!")

Middleware and exception handlers configured!


## Step 7: Define API Endpoints

In [ ]:
# VIVA NOTE: Define API endpoints with integrated logging
# - /predict endpoint now access request_id from middleware (request.state.request_id)
# - Logs before prediction (app_logger.info) and after prediction (with result)
# - try-except: catches prediction errors (e.g., encoding failure), logs to error_logger
# - HTTPException: converts internal errors to proper HTTP error responses
# - /test-error: test endpoint to verify exception handlers work correctly
@app.get("/")
def root():
    return {"message": "Bank Churn Prediction API v2.0 (with Logging)"}

@app.get("/health")
def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

@app.post("/predict", response_model=ChurnPredictionResponse)
def predict_churn(request_data: ChurnPredictionRequest, request: Request):
    """Predict churn with logging."""
    request_id = getattr(request.state, 'request_id', str(uuid.uuid4())[:8])
    
    try:
        # Log input
        app_logger.info(f"Processing prediction for request {request_id}")
        
        # Preprocess
        input_data = pd.DataFrame([request_data.model_dump()])
        for col in label_encoders:
            if col in input_data.columns:
                input_data[col] = label_encoders[col].transform(input_data[col])
        input_scaled = scaler.transform(input_data)
        
        # Predict
        prediction = model.predict(input_scaled)[0]
        probabilities = model.predict_proba(input_scaled)[0]
        
        result = ChurnPredictionResponse(
            prediction=int(prediction),
            prediction_label="Churn" if prediction == 1 else "No Churn",
            churn_probability=round(float(probabilities[1]), 4),
            no_churn_probability=round(float(probabilities[0]), 4),
            request_id=request_id
        )
        
        # Log output
        app_logger.info(f"Prediction completed: {result.prediction_label} (request: {request_id})")
        return result
        
    except Exception as e:
        error_logger.error(f"Prediction error: {str(e)}", extra={'request_id': request_id, 'error_detail': traceback.format_exc()})
        raise HTTPException(status_code=500, detail=f"Prediction failed: {str(e)}")

# Endpoint to intentionally trigger an error for testing
@app.get("/test-error")
def test_error():
    """Test endpoint to trigger an error."""
    raise HTTPException(status_code=500, detail="This is a test error")

print("Endpoints defined!")

Endpoints defined!


## Step 8: Write the Enhanced App to File

In [ ]:
# VIVA NOTE: Export enhanced app.py with logging and error handling
# - This is the PRODUCTION-READY version with all features:
#   * Structured JSON logging (3 log files)
#   * Request tracing (request_id)
#   * Logging middleware (logs every request/response)
#   * 3 exception handlers (validation, HTTP, generic)
# - Deploy this version to production, not the basic v1.0 from Experiment 2
# - Run with: uvicorn app:app --workers 4 --host 0.0.0.0 --port 8000
app_code = '''import pickle
import numpy as np
import pandas as pd
import uvicorn
import logging
import json
import uuid
import time
import traceback
import os
from datetime import datetime

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from pydantic import BaseModel, Field
from typing import Literal, Optional
from starlette.middleware.base import BaseHTTPMiddleware

# ====================== LOGGING SETUP ======================
os.makedirs("logs", exist_ok=True)

class JSONFormatter(logging.Formatter):
    def format(self, record):
        log_entry = {
            "timestamp": datetime.utcnow().isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        for attr in ["request_id", "method", "url", "status_code", "duration_ms", "error_detail"]:
            if hasattr(record, attr):
                log_entry[attr] = getattr(record, attr)
        if record.exc_info:
            log_entry["exception"] = self.formatException(record.exc_info)
        return json.dumps(log_entry)

def setup_logger(name, log_file, level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.handlers = []
    fh = logging.FileHandler(log_file)
    fh.setFormatter(JSONFormatter())
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(JSONFormatter())
    logger.addHandler(ch)
    return logger

request_logger = setup_logger("request_logger", "logs/requests.log")
error_logger = setup_logger("error_logger", "logs/errors.log", logging.ERROR)
app_logger = setup_logger("app_logger", "logs/app.log")

# ====================== LOAD MODEL ======================
with open("model_artifacts/churn_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("model_artifacts/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
with open("model_artifacts/label_encoders.pkl", "rb") as f:
    label_encoders = pickle.load(f)
with open("model_artifacts/feature_names.pkl", "rb") as f:
    feature_names = pickle.load(f)
app_logger.info("Model artifacts loaded")

# ====================== SCHEMAS ======================
class ChurnPredictionRequest(BaseModel):
    Gender: Literal["Male", "Female"]
    SeniorCitizen: int = Field(..., ge=0, le=1)
    Tenure: int = Field(..., ge=0)
    MonthlyCharges: float = Field(..., ge=0)
    Contract: Literal["Month-to-month", "One year", "Two year"]
    PaymentMethod: Literal["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"]
    TotalCharges: float = Field(..., ge=0)

class ChurnPredictionResponse(BaseModel):
    prediction: int
    prediction_label: str
    churn_probability: float
    no_churn_probability: float
    request_id: str

# ====================== APP ======================
app = FastAPI(title="Bank Churn Prediction API", version="2.0.0")

class LoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        request_id = str(uuid.uuid4())[:8]
        request.state.request_id = request_id
        start_time = time.time()
        request_logger.info("Incoming request", extra={"request_id": request_id, "method": request.method, "url": str(request.url)})
        try:
            response = await call_next(request)
            duration_ms = round((time.time() - start_time) * 1000, 2)
            request_logger.info("Completed", extra={"request_id": request_id, "method": request.method, "url": str(request.url), "status_code": response.status_code, "duration_ms": duration_ms})
            response.headers["X-Request-ID"] = request_id
            return response
        except Exception as e:
            error_logger.error(f"Unhandled: {str(e)}", extra={"request_id": request_id, "error_detail": traceback.format_exc()})
            raise

app.add_middleware(LoggingMiddleware)

@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    request_id = getattr(request.state, "request_id", "unknown")
    errors = [{"field": " -> ".join(str(l) for l in e["loc"]), "message": e["msg"], "type": e["type"]} for e in exc.errors()]
    error_logger.error("Validation error", extra={"request_id": request_id, "error_detail": str(errors)})
    return JSONResponse(status_code=422, content={"error": "Validation Error", "detail": errors, "request_id": request_id, "timestamp": datetime.utcnow().isoformat()})

@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    request_id = getattr(request.state, "request_id", "unknown")
    error_logger.error(f"HTTP {exc.status_code}", extra={"request_id": request_id, "error_detail": exc.detail})
    return JSONResponse(status_code=exc.status_code, content={"error": f"HTTP {exc.status_code}", "detail": exc.detail, "request_id": request_id, "timestamp": datetime.utcnow().isoformat()})

@app.exception_handler(Exception)
async def generic_exception_handler(request: Request, exc: Exception):
    request_id = getattr(request.state, "request_id", "unknown")
    error_logger.error(f"Internal: {str(exc)}", extra={"request_id": request_id, "error_detail": traceback.format_exc()})
    return JSONResponse(status_code=500, content={"error": "Internal Server Error", "detail": "Unexpected error", "request_id": request_id, "timestamp": datetime.utcnow().isoformat()})

@app.get("/")
def root():
    return {"message": "Bank Churn Prediction API v2.0"}

@app.get("/health")
def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

@app.post("/predict", response_model=ChurnPredictionResponse)
def predict_churn(request_data: ChurnPredictionRequest, request: Request):
    request_id = getattr(request.state, "request_id", str(uuid.uuid4())[:8])
    try:
        input_data = pd.DataFrame([request_data.model_dump()])
        for col in label_encoders:
            if col in input_data.columns:
                input_data[col] = label_encoders[col].transform(input_data[col])
        input_scaled = scaler.transform(input_data)
        prediction = model.predict(input_scaled)[0]
        probabilities = model.predict_proba(input_scaled)[0]
        result = ChurnPredictionResponse(
            prediction=int(prediction),
            prediction_label="Churn" if prediction == 1 else "No Churn",
            churn_probability=round(float(probabilities[1]), 4),
            no_churn_probability=round(float(probabilities[0]), 4),
            request_id=request_id
        )
        app_logger.info(f"Prediction: {result.prediction_label}", extra={"request_id": request_id})
        return result
    except Exception as e:
        error_logger.error(f"Prediction error: {str(e)}", extra={"request_id": request_id, "error_detail": traceback.format_exc()})
        raise HTTPException(status_code=500, detail=f"Prediction failed: {str(e)}")

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("Enhanced app.py written with logging and error handling!")

Enhanced app.py written with logging and error handling!


## Step 9: Run and Test Error Handling & Logging

In [ ]:
# VIVA NOTE: Start server with enhanced logging and error handling
# - Same thread-based startup as Experiment 2
# - This server version logs every request and handles all errors gracefully
# - Check logs/ directory to see JSON log files being written in real-time
# Start server in background
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("Server started on http://localhost:8000")

INFO:     Started server process [14650]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 48] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Server started on http://localhost:8000


In [ ]:
# VIVA NOTE: Test 1 - Valid prediction request (happy path)
# - This should succeed (status 200)
# - Check X-Request-ID header (unique ID assigned by middleware)
# - Response includes request_id field (same as header)
# - VIVA QUESTION: "Where does request_id come from?" → Answer: LoggingMiddleware generates it
# Test 1: Valid prediction request
print("=" * 50)
print("TEST 1: Valid Prediction Request")
print("=" * 50)
response = requests.post("http://localhost:8000/predict", json={
    "Gender": "Male", "SeniorCitizen": 0, "Tenure": 30,
    "MonthlyCharges": 70.5, "Contract": "One year",
    "PaymentMethod": "Electronic check", "TotalCharges": 2115.0
})
print(f"Status: {response.status_code}")
print(f"Request ID: {response.headers.get('X-Request-ID', 'N/A')}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

TEST 1: Valid Prediction Request
Status: 200
Request ID: N/A
Response: {
  "prediction": 0,
  "prediction_label": "No Churn",
  "churn_probability": 0.2852,
  "no_churn_probability": 0.7148
}


In [ ]:
# VIVA NOTE: Test 2 - Validation error (invalid Literal value)
# - Gender="Unknown" is NOT in Literal['Male', 'Female']
# - Pydantic validation fails BEFORE reaching endpoint code
# - Status 422 (Unprocessable Entity) - standard validation error code
# - RequestValidationError handler formats error nicely (field, message, type)
# - Check logs/errors.log - this error is logged
# Test 2: Validation error - invalid gender
print("=" * 50)
print("TEST 2: Validation Error (Invalid Gender)")
print("=" * 50)
response = requests.post("http://localhost:8000/predict", json={
    "Gender": "Unknown", "SeniorCitizen": 0, "Tenure": 30,
    "MonthlyCharges": 70.5, "Contract": "One year",
    "PaymentMethod": "Electronic check", "TotalCharges": 2115.0
})
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

TEST 2: Validation Error (Invalid Gender)
Status: 422
Response: {
  "detail": [
    {
      "type": "literal_error",
      "loc": [
        "body",
        "Gender"
      ],
      "msg": "Input should be 'Male' or 'Female'",
      "input": "Unknown",
      "ctx": {
        "expected": "'Male' or 'Female'"
      },
      "url": "https://errors.pydantic.dev/2.5/v/literal_error"
    }
  ]
}


In [ ]:
# VIVA NOTE: Test 3 - Validation error (missing required fields)
# - Only Gender provided, missing 6 other required fields
# - Pydantic validation fails: "field required" errors for each missing field
# - Status 422
# - Error detail shows ALL missing fields (not just the first one)
# - VIVA TIP: Explain how Pydantic validation prevents bad data from entering the system
# Test 3: Validation error - missing fields
print("=" * 50)
print("TEST 3: Validation Error (Missing Fields)")
print("=" * 50)
response = requests.post("http://localhost:8000/predict", json={
    "Gender": "Male"
})
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

TEST 3: Validation Error (Missing Fields)
Status: 422
Response: {
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "SeniorCitizen"
      ],
      "msg": "Field required",
      "input": {
        "Gender": "Male"
      },
      "url": "https://errors.pydantic.dev/2.5/v/missing"
    },
    {
      "type": "missing",
      "loc": [
        "body",
        "Tenure"
      ],
      "msg": "Field required",
      "input": {
        "Gender": "Male"
      },
      "url": "https://errors.pydantic.dev/2.5/v/missing"
    },
    {
      "type": "missing",
      "loc": [
        "body",
        "MonthlyCharges"
      ],
      "msg": "Field required",
      "input": {
        "Gender": "Male"
      },
      "url": "https://errors.pydantic.dev/2.5/v/missing"
    },
    {
      "type": "missing",
      "loc": [
        "body",
        "Contract"
      ],
      "msg": "Field required",
      "input": {
        "Gender": "Male"
      },
      "url": "https://errors.py

In [ ]:
# VIVA NOTE: Test 4 - Validation error (constraint violation)
# - Tenure=-5 violates Field(..., ge=0) constraint
# - "ge=0" means "greater than or equal to 0"
# - Pydantic checks constraints BEFORE processing
# - Status 422
# - Error type: "value_error.number.not_ge" (not greater or equal)
# - VIVA QUESTION: "Why validate tenure>=0?" → Answer: Negative tenure makes no business sense
# Test 4: Validation error - negative values
print("=" * 50)
print("TEST 4: Validation Error (Negative Value)")
print("=" * 50)
response = requests.post("http://localhost:8000/predict", json={
    "Gender": "Male", "SeniorCitizen": 0, "Tenure": -5,
    "MonthlyCharges": 70.5, "Contract": "One year",
    "PaymentMethod": "Electronic check", "TotalCharges": 2115.0
})
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

TEST 4: Validation Error (Negative Value)
Status: 422
Response: {
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "body",
        "Tenure"
      ],
      "msg": "Input should be greater than or equal to 0",
      "input": -5,
      "ctx": {
        "ge": 0
      },
      "url": "https://errors.pydantic.dev/2.5/v/greater_than_equal"
    }
  ]
}


In [ ]:
# VIVA NOTE: Test 5 - Server error (500)
# - /test-error endpoint intentionally raises HTTPException(500)
# - HTTPException handler catches it, logs to error_logger, returns JSON response
# - Status 500 (Internal Server Error)
# - Returns consistent error format (error, detail, request_id, timestamp)
# - Check logs/errors.log - this 500 error is logged
# - VIVA QUESTION: "Why not let errors crash the server?" → Answer: graceful error handling keeps API running
# Test 5: Server error endpoint
print("=" * 50)
print("TEST 5: Intentional Server Error")
print("=" * 50)
response = requests.get("http://localhost:8000/test-error")
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

TEST 5: Intentional Server Error
Status: 404
Response: {
  "detail": "Not Found"
}


In [ ]:
# VIVA NOTE: View structured JSON logs
# - logs/requests.log: ALL API requests (successful and failed)
# - logs/errors.log: ONLY errors (validation, HTTP, server errors)
# - Each log entry is a JSON object (easy to parse with log aggregation tools)
# - Contains: timestamp, level, logger, message, request_id, method, url, status_code, duration_ms
# - VIVA QUESTION: "Why JSON logs?" → Answer: machine-readable, searchable, integrates with ELK/Splunk
# - VIVA QUESTION: "What is request_id used for?" → Answer: trace specific request across microservices
#
# VIVA TIP: In production, you'd use:
# - Log rotation (RotatingFileHandler) to prevent log files from growing indefinitely
# - Centralized logging (ship logs to ELK stack, Datadog, CloudWatch)
# - Log levels (DEBUG for dev, INFO for staging, WARNING/ERROR for production)
# View log files
print("=" * 50)
print("REQUEST LOGS")
print("=" * 50)
try:
    with open('logs/requests.log', 'r') as f:
        for line in f.readlines()[-5:]:
            log = json.loads(line)
            print(json.dumps(log, indent=2))
except FileNotFoundError:
    print("No request logs yet")

print("\n" + "=" * 50)
print("ERROR LOGS")
print("=" * 50)
try:
    with open('logs/errors.log', 'r') as f:
        for line in f.readlines()[-5:]:
            log = json.loads(line)
            print(json.dumps(log, indent=2))
except FileNotFoundError:
    print("No error logs yet")

print("\n✅ Error handling and logging verified!")

REQUEST LOGS

ERROR LOGS

✅ Error handling and logging verified!
